# Creating Bronze Layer


## Connect my spark cluster to my Azure datalake 

## Create Delta Tables from .csv files

In [0]:
#df = spark.read \
#    .option("header", "true") \
#    .option("inferSchema", "true") \
#    .csv("abfss://olist-ecommerce@datalakemaster.dfs.core.windows.net/landing/customers.csv")

#df.show()

In [0]:
from pyspark.sql.functions import current_timestamp, col

def createBronzeDeltaTable(storage_account, storage_container, landing_file_name, delta_table_name):
    landing_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/landing"
    bronze_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/bronze"

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"{landing_path}/{landing_file_name}")
    
    # sada dodajemo stupce za timestamp i source_file_name 
    df_with_ts_and_source = df \
        .withColumn("ingestion_timestamp", current_timestamp()) \
        .withColumn("source_file_name", col("_metadata.file_path"))
    
    df_with_ts_and_source.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{bronze_path}/{delta_table_name}")

    # sanity check
    sanity_df = spark.read.format("delta").load(f"{bronze_path}/{delta_table_name}")
    sanity_df.show()
    sanity_df.printSchema()

In [0]:
storage_account = "datalakemaster"
storage_container = "olist-ecommerce"

tables = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

for delta_table_name, landing_file_name in tables.items():
    createBronzeDeltaTable(storage_account, storage_container, landing_file_name, delta_table_name)